In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt


In [3]:
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split


In [4]:
mask_files=os.listdir(r'C:\Users\kanha\Downloads\archive (4)\data\with_mask')
print(mask_files[0:5])


['with_mask_1.jpg', 'with_mask_10.jpg', 'with_mask_100.jpg', 'with_mask_1000.jpg', 'with_mask_1001.jpg']


In [5]:
without_mask=os.listdir(r"C:\Users\kanha\Downloads\archive (4)\data\without_mask")
print(without_mask[0:5])
print(len(without_mask),len(mask_files))

['without_mask_1.jpg', 'without_mask_10.jpg', 'without_mask_100.jpg', 'without_mask_1000.jpg', 'without_mask_1001.jpg']
3828 3725


In [ ]:
import os

mask_dir = r'C:\Users\kanha\Downloads\archive (4)\data\with_mask'
without_dir = r'C:\Users\kanha\Downloads\archive (4)\data\without_mask'

mask_files = os.listdir(mask_dir)
without_mask = os.listdir(without_dir)

img_list = []
labels = []
for file in mask_files:
    img_list.append(os.path.join(mask_dir, file))
    labels.append(1)


for file in without_mask:
    img_list.append(os.path.join(without_dir, file))
    labels.append(0)

print(img_list[:5])

['C:\\Users\\kanha\\Downloads\\archive (4)\\data\\with_mask\\with_mask_1.jpg', 'C:\\Users\\kanha\\Downloads\\archive (4)\\data\\with_mask\\with_mask_10.jpg', 'C:\\Users\\kanha\\Downloads\\archive (4)\\data\\with_mask\\with_mask_100.jpg', 'C:\\Users\\kanha\\Downloads\\archive (4)\\data\\with_mask\\with_mask_1000.jpg', 'C:\\Users\\kanha\\Downloads\\archive (4)\\data\\with_mask\\with_mask_1001.jpg']


In [7]:
%pip install torchvision

Note: you may need to restart the kernel to use updated packages.


In [8]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    
])

In [9]:
class custom(Dataset):
    def __init__(self,img_list,labels,transform=None):
        self.img_list=img_list
        self.labels=labels
        self.transform=transform
        
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self,index):
        img_path=self.img_list[index]
        label=self.labels[index]
        image=Image.open(img_path)
        if self.transform:
            image=self.transform(image)
        return image,label


In [10]:
#labels
with_mask_label=[1]*3725
without_mask_label=[0]*3828

In [11]:
labels=with_mask_label+without_mask_label
print(labels[0:5])

[1, 1, 1, 1, 1]


In [12]:
len(labels)

7553

In [13]:
dataset=custom(img_list,labels,transform=transform)


In [14]:
from torch.utils.data import random_split
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size
train_data,val_data=random_split(dataset,[train_size,test_size])

In [15]:
img, label = train_data[0]
print(img.shape)

torch.Size([3, 128, 128])


In [16]:
len(train_data)

6042

In [17]:
torch.manual_seed(42)

In [18]:
train_loader=DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)
val_loader=DataLoader(
    val_data,
     batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [19]:
class mymodel(nn.Module):
    def __init__(self,n_features):
        super().__init__()
        self.model=nn.Sequential(
            nn.Conv2d(n_features,32,kernel_size=3,padding=1,stride=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(32,64,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64,2)


        )
    def forward(self,x):
        return self.model(x)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = mymodel(3).to(device)


In [21]:
lossfn=nn.CrossEntropyLoss()
optim=torch.optim.Adam(model.parameters(),lr=0.001)
from tqdm import tqdm

In [22]:
epochs=10


model.train()
for epoch in range(epochs):
    total_loss=0
    correct=0
    total=0
    for X,y in tqdm(train_loader,desc=f"Epoch {epoch+1}"):
        X, y = X.to(device), y.to(device)
        output=model(X)
        loss=lossfn(output,y)
        optim.zero_grad()
        loss.backward()
        optim.step()
        _, preds = torch.max(output, dim=1)
        total_loss+=loss.item()
        total+=X.size(0)
        correct += (preds == y).sum().item()
    acc=correct/total
    print(f"Loss:{total_loss} Accuracy={acc}")
    

Epoch 1:  25%|██▍       | 47/189 [00:37<01:49,  1.30it/s]c:\Users\kanha\anaconda3\Lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1: 100%|██████████| 189/189 [02:26<00:00,  1.29it/s]


Loss:93.82444140315056 Accuracy=0.7664680569347898


Epoch 2: 100%|██████████| 189/189 [01:36<00:00,  1.96it/s]


Loss:77.79962247610092 Accuracy=0.8154584574644158


Epoch 3: 100%|██████████| 189/189 [01:48<00:00,  1.74it/s]


...

Loss:35.114446334540844 Accuracy=0.9314796425024826


Epoch 9: 100%|██████████| 189/189 [01:25<00:00,  2.21it/s]


Loss:35.847260899841785 Accuracy=0.9286660046342271


Epoch 10: 100%|██████████| 189/189 [01:25<00:00,  2.20it/s]

Loss:31.501727996394038 Accuracy=0.9376034425686859


In [23]:
model.eval()

mymodel(
  (model): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (12): AdaptiveAvgPool2d(output_size=(1, 1))
    (13): Flatten(start_dim=1, end_dim=-1)
    (14): Linear(in_features=128, out_features=64, b

In [24]:
total=0

correct=0
with torch.no_grad():
    for x,y in tqdm(val_loader):
        x,y=x.to(device),y.to(device)
        output=model(x)
        predict=torch.argmax(output,dim=1)
        correct+=(predict==y).sum().item()
        total+=x.size(0)
acc=correct/total
print(acc)


100%|██████████| 48/48 [00:21<00:00,  2.23it/s]

0.8696227663798809


In [25]:


%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [30]:
from torchinfo import summary
summary(model ,input_size=(1,3,128,128))

Layer (type:depth-idx)                   Output Shape              Param #
mymodel                                  [1, 2]                    --
├─Sequential: 1-1                        [1, 2]                    --
│    └─Conv2d: 2-1                       [1, 32, 128, 128]         896
│    └─BatchNorm2d: 2-2                  [1, 32, 128, 128]         64
│    └─ReLU: 2-3                         [1, 32, 128, 128]         --
│    └─MaxPool2d: 2-4                    [1, 32, 64, 64]           --
│    └─Conv2d: 2-5                       [1, 64, 64, 64]           18,496
│    └─BatchNorm2d: 2-6                  [1, 64, 64, 64]           128
│    └─ReLU: 2-7                         [1, 64, 64, 64]           --
│    └─MaxPool2d: 2-8                    [1, 64, 32, 32]           --
│    └─Conv2d: 2-9                       [1, 128, 32, 32]          73,856
│    └─BatchNorm2d: 2-10                 [1, 128, 32, 32]          256
│    └─ReLU: 2-11                        [1, 128, 32, 32]          --
│   

In [31]:
def predict_image(img_path,transform,model,device):
    image=Image.open(img_path).convert("RGB")
    image=transform(image)
    image=image.unsqueeze(0)
    image=image.to(device)
    with torch.no_grad():
        output=model(image)
        _,pred=torch.max(output,1)
    return pred.item()

In [39]:
ig_path=r"C:\Users\kanha\Downloads\archive (4)\data\maw.jpeg"
print("with mask" if predict_image(img_path=ig_path,transform=transform,model=model,device=device)==1 else "No mask")

with mask
